# Прогнозирование расхода топлива автомобиля

### Обработка данных

## Данные: Auto MPG

Целевая переменная — **MPG** (миль на галлон, расход топлива), признаки — характеристики авто (цилиндры, объём, мощность, вес и т.д.). Задача — регрессия.

In [15]:
import pandas as pd 

In [16]:
url = 'http://archive.ics.uci.edu/ml/' \
'machine-learning-databases/auto-mpg/auto-mpg.data'

column_names =  ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight', 
                 'Acceleration', 'Model Year', 'Origin']

df = pd.read_csv(url, names = column_names, na_values = '?', comment = '\t', 
                 sep = ' ', skipinitialspace = True)

df.head()

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
0,18.0,8,307.0,130.0,3504.0,12.0,70,1
1,15.0,8,350.0,165.0,3693.0,11.5,70,1
2,18.0,8,318.0,150.0,3436.0,11.0,70,1
3,16.0,8,304.0,150.0,3433.0,12.0,70,1
4,17.0,8,302.0,140.0,3449.0,10.5,70,1


## Пропуски

Horsepower: 392 значения вместо 398 — есть пропуски (в файле помечены как '?'). Уберём их через `dropna()`.

In [17]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
MPG,398.0,23.514573,7.815984,9.0,17.500,23.0,29.000,46.6
Cylinders,398.0,5.454774,1.701004,3.0,4.000,4.0,8.000,8.0
Displacement,398.0,193.425879,104.269838,68.0,104.250,148.5,262.000,455.0
Horsepower,392.0,104.469388,38.491160,46.0,75.000,93.5,126.000,230.0
Weight,398.0,2970.424623,846.841774,1613.0,2223.750,2803.5,3608.000,5140.0
Acceleration,398.0,15.568090,2.757689,8.0,13.825,15.5,17.175,24.8
Model Year,398.0,76.010050,3.697627,70.0,73.000,76.0,79.000,82.0
Origin,398.0,1.572864,0.802055,1.0,1.000,1.0,2.000,3.0


In [18]:
df = df.dropna()
df = df.reset_index(drop = True)

## Подготовка данных

- Сплит 80/20 с фиксированным seed.
- Z-нормализация числовых признаков по mean/std, посчитанным только по train — чтобы тест не «подглядывал» в статистику обучающей выборки.

In [19]:
import sklearn
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, train_size=0.8, random_state=42)
train_stats = df_train.describe().transpose()

numeric_column_names = [
    'Cylinders', 'Displacement',
    'Horsepower', 'Weight',
    'Acceleration'
]

df_train_norm, df_test_norm = df_train.copy(), df_test.copy()

for соl_name in numeric_column_names: 
    df_train_norm[соl_name] = df_train_norm[соl_name].astype(float)
    df_test_norm[соl_name] = df_test_norm[соl_name].astype(float)

    mean = train_stats.loc[соl_name, 'mean']
    std = train_stats.loc[соl_name, 'std']
    df_train_norm.loc[:, соl_name] = (df_train_norm.loc[:, соl_name] - mean)/std    
    df_test_norm.loc[:, соl_name] = (df_test_norm.loc[:, соl_name] - mean)/std     

In [20]:
df_train_norm.tail() 

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
71,15.0,1.480536,1.045447,1.186034,1.076969,-1.080283,72,1
106,18.0,0.304374,0.351582,-0.120005,-0.234356,-0.193086,73,1
270,23.8,-0.871788,-0.429017,-0.511817,-0.155890,0.729599,78,1
348,29.9,-0.871788,-0.939778,-1.034233,-0.720604,1.829723,81,1
102,11.0,1.480536,1.970600,1.186034,2.390672,-0.547965,73,1


Сгруппируем информацию о годе производства в сегменты

In [22]:
import torch

boundaries = torch.tensor([73, 76, 79]) 

v = torch.tensor(df_train_norm['Model Year'].values)
df_train_norm['Model Year Bucketed'] = torch.bucketize(v, boundaries, right=True)

v = torch.tensor(df_test_norm['Model Year'].values)
df_test_norm['Model Year Bucketed'] = torch.bucketize(v, boundaries, right=True)

numeric_column_names.append('Model Year Bucketed')


Применим метод унитарного кодирования для категориального признака, чтобы преобразовать его в плотный формат:

In [24]:
from torch.nn.functional import one_hot

total_origin = len(set(df_train_norm['Origin'])) 

origin_encoded = one_hot(torch.from_numpy(df_train_norm['Origin'].values) % total_origin) 
x_train_numeric = torch.tensor(df_train_norm[numeric_column_names].values)
x_train = torch.cat([x_train_numeric, origin_encoded], 1).float()

origin_encoded = one_hot(torch.from_numpy(df_test_norm['Origin'].values) % total_origin) 
x_test_numeric = torch.tensor(df_test_norm[numeric_column_names].values) 
x_test = torch.cat([x_test_numeric, origin_encoded], 1).float() 

y_train = torch.tensor(df_train_norm['MPG'].values).float()
y_test = torch.tensor(df_test_norm['MPG'].values).float()

### Обучение регрессионной модели DNN

In [27]:
from torch.utils.data import TensorDataset, DataLoader
from torch import nn

train_ds = TensorDataset(x_train, y_train)
batch_size = 8
torch.manual_seed(1)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

## Архитектура

Полносвязная сеть: скрытые слои 8 и 4 с ReLU, выход — 1 нейрон без активации (регрессия предсказывает MPG напрямую).

In [28]:
hidden_units = [8, 4]
input_size = x_train.shape[1]
all_layers = [] 

for hidden_unit in hidden_units:
    layer = nn.Linear(input_size, hidden_unit)
    all_layers.append(layer)
    all_layers.append(nn.ReLU())
    input_size = hidden_unit 

all_layers.append(nn.Linear(hidden_units[-1], 1))
model = nn.Sequential(*all_layers)

## Loss и оптимизатор

Для регрессии — `MSELoss`. Оптимизатор SGD (lr=0.001).

In [29]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001) 

## Цикл обучения

Тот же паттерн: forward → loss → backward → step → zero_grad. Раз в 20 эпох печатается средний по батчам loss.

In [32]:
torch.manual_seed(1)
num_epochs = 300
log_epochs = 20
for epoch in range(num_epochs):
    loss_hist_train = 0
    for x_batch, y_batch in train_dl:
        pred = model(x_batch)[:, 0]
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        loss_hist_train += loss.item()

    if epoch % log_epochs == 0:
        print(f'Эпoxa {epoch} Потеря {loss_hist_train / len(train_dl)}')

Эпoxa 0 Потеря 7.973310321569443
Эпoxa 20 Потеря 7.624822437763214
Эпoxa 40 Потеря 8.66631527096033
Эпoxa 60 Потеря 7.721397860348224
Эпoxa 80 Потеря 8.495815141499042
Эпoxa 100 Потеря 9.735792122781277
Эпoxa 120 Потеря 7.422363725819741
Эпoxa 140 Потеря 8.334280741214751
Эпoxa 160 Потеря 7.803904187679291
Эпoxa 180 Потеря 7.93619539886713
Эпoxa 200 Потеря 7.76283995732665
Эпoxa 220 Потеря 7.635651160753332
Эпoxa 240 Потеря 7.839635632932186
Эпoxa 260 Потеря 7.527923107147217
Эпoxa 280 Потеря 7.394112101197242


## Результаты

На отложенном тесте: MSE ≈ 10.9, MAE ≈ 2.45 — средняя ошибка предсказания ~2.5 миль на галлон.

In [33]:
with torch.no_grad():
    pred = model(x_test.float())[:, 0] 
    loss = loss_fn(pred, y_test)
    print(f'MSE при тестировании: {loss.item():.4f}') 
    print(f'МAE при тестировании: {nn.L1Loss()(pred, y_test).item():.4f}') 


MSE при тестировании: 10.9220
МAE при тестировании: 2.4549
